In [1]:
import fairlearn
import transformers
import kaggle
import matplotlib
print("All packages working")


C:\Users\nikhil.malige.RHYMTECH\.conda\envs\fairlearn-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


OSError: Could not find kaggle.json. Make sure it's located in C:\Users\nikhil.malige.RHYMTECH\.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/

In [ ]:
import fairlearn
import transformers
import matplotlib
print("Core packages working")


In [ ]:
import os

# Set your Kaggle credentials (from kaggle.com → Account → API)
os.environ['KAGGLE_USERNAME'] = 'nikhilmalige'
os.environ['KAGGLE_KEY'] = 'KGAT_ddc9c90520413c561ef153bacb00ab41'

# Download dataset
import kaggle
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    'fedesoriano/stroke-prediction-dataset',
    path='./data',
    unzip=True
)

In [ ]:
import os
# List downloaded files
for f in os.listdir('./data'):
    print(f)
    

In [ ]:
import os
print(os.path.abspath('./data'))

In [ ]:
from huggingface_hub import hf_hub_download
from sklearn.ensemble import GradientBoostingClassifier

# Define the model (sklearn-compatible, works with Fairlearn)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)

print("Model ready ✓")
print(model)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv('./data/healthcare-dataset-stroke-data.csv')

# Drop nulls
df.dropna(inplace=True)

# Drop irrelevant column
df.drop(columns=['id'], inplace=True)

# Encode categorical columns
le = LabelEncoder()
for col in ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']:
    df[col] = le.fit_transform(df[col])

# Sensitive feature for Fairlearn (gender or age)
sensitive_features = df['gender']

# Features and target
X = df.drop(columns=['stroke'])
y = df['stroke']

# Train/test split
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_features,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Training complete ✓")
print(f"Test samples: {len(y_test)}")
print(f"Predictions done: {len(y_pred)}")

In [ ]:
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate
)
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- MetricFrame: Core Fairlearn Audit ---
metrics_dict = {
    "accuracy": accuracy_score,
    "precision": lambda y_true, y_pred: precision_score(y_true, y_pred, zero_division=0),
    "recall": lambda y_true, y_pred: recall_score(y_true, y_pred, zero_division=0),
    "selection_rate": selection_rate
}

mf = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test
)

print("=== Overall Metrics ===")
print(mf.overall)

print("\n=== Metrics by Gender Group ===")
print(mf.by_group)

# --- Bias Disparity Scores ---
dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=sf_test)
eod = equalized_odds_difference(y_test, y_pred, sensitive_features=sf_test)

print(f"\nDemographic Parity Difference : {dpd:.4f}  (0 = perfectly fair)")
print(f"Equalized Odds Difference     : {eod:.4f}  (0 = perfectly fair)")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Fix class imbalance
weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}

# Retrain with balanced weights
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
sample_weights = y_train.map(class_weight_dict)
model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

print("Retrained with class balancing ✓")

In [ ]:
mf = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test
)

print("=== Overall Metrics ===")
print(mf.overall)

print("\n=== Metrics by Gender Group ===")
print(mf.by_group)

dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=sf_test)
eod = equalized_odds_difference(y_test, y_pred, sensitive_features=sf_test)

print(f"\nDemographic Parity Difference : {dpd:.4f}  (0 = perfectly fair)")
print(f"Equalized Odds Difference     : {eod:.4f}  (0 = perfectly fair)")

In [ ]:
import matplotlib.pyplot as plt

mf.by_group.plot(
    kind='bar',
    subplots=True,
    layout=(2, 2),
    figsize=(12, 8),
    title="Fairness Metrics by Gender Group"
)
plt.suptitle("Fairlearn Audit — Stroke Prediction Model", fontsize=14)
plt.tight_layout()
plt.savefig('./fairness_audit.png', dpi=150)
plt.show()
print("Visualization saved ✓")

In [ ]:
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

# Bias mitigation
mitigator = ExponentiatedGradient(
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
    constraints=DemographicParity()
)
mitigator.fit(X_train, y_train, sensitive_features=sf_train)
y_pred_mitigated = mitigator.predict(X_test)

# Re-audit after mitigation
dpd_post = demographic_parity_difference(
    y_test, y_pred_mitigated, sensitive_features=sf_test
)
eod_post = equalized_odds_difference(
    y_test, y_pred_mitigated, sensitive_features=sf_test
)

print(f"Before Mitigation — DPD: 0.0451 | EOD: 0.0651")
print(f"After  Mitigation — DPD: {dpd_post:.4f} | EOD: {eod_post:.4f}")

In [ ]:
import urllib.request, os

# Write the UI file directly to your project folder
url = "https://raw.githubusercontent.com/streamlit/streamlit/develop/README.md"  # test only

# Just write the file content directly
ui_content = open("fairlearn_ui.py", "w", encoding="utf-8")
ui_content.write("""
# paste nothing - we write it below
""")
ui_content.close()

print(os.path.abspath("fairlearn_ui.py"))

In [ ]:
import os

# Check where we are
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

In [ ]:
code = r'''import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, pickle
warnings.filterwarnings('ignore')

st.set_page_config(page_title="AI Model Fairness Audit Platform", page_icon="⚕️", layout="wide")
st.markdown("""<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@400;500;600&display=swap');
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;}
section[data-testid="stSidebar"]{background:#0d1117;border-right:1px solid #21262d;}
section[data-testid="stSidebar"] *{color:#e6edf3 !important;}
.main .block-container{padding:1.5rem 2rem;max-width:1400px;}
.step-card{background:#161b22;border:1px solid #21262d;border-radius:12px;padding:1.8rem;margin-bottom:1rem;}
.step-card.active{border-color:#238636;} .step-card.done{border-color:#1f6feb;}
.step-num{font-family:'Space Mono',monospace;font-size:0.7rem;color:#8b949e;letter-spacing:2px;text-transform:uppercase;}
.step-title{font-family:'Space Mono',monospace;font-size:1rem;font-weight:700;color:#e6edf3;margin:0.3rem 0;}
.step-desc{font-size:0.83rem;color:#8b949e;}
.mcard{background:#0d1117;border:1px solid #21262d;border-radius:10px;padding:1.2rem;text-align:center;}
.mval{font-family:'Space Mono',monospace;font-size:1.8rem;font-weight:700;color:#58a6ff;}
.mval.g{color:#56d364;}.mval.o{color:#ffa657;}.mval.r{color:#f85149;}
.mlbl{font-size:0.72rem;color:#8b949e;text-transform:uppercase;letter-spacing:1px;margin-top:4px;}
.sbox{background:#0d1117;border:1px solid #21262d;border-radius:8px;padding:1rem 1.5rem;
      font-family:'Space Mono',monospace;font-size:0.8rem;color:#8b949e;line-height:2;}
.shdr{font-family:'Space Mono',monospace;font-size:0.8rem;color:#8b949e;text-transform:uppercase;
      letter-spacing:2px;border-bottom:1px solid #21262d;padding-bottom:8px;margin:1.5rem 0 1rem;}
div[data-testid="stButton"]>button{background:#238636;color:#fff;border:none;border-radius:8px;font-weight:600;width:100%;}
div[data-testid="stButton"]>button:hover{background:#2ea043;}
#MainMenu,footer,header{visibility:hidden;}
</style>""", unsafe_allow_html=True)

for k,v in [("step",1),("model_obj",None),("model_name",""),("trained",False),("train_results",{}),("audit_done",False)]:
    if k not in st.session_state: st.session_state[k] = v

st.markdown("""<div style="background:linear-gradient(135deg,#0d1117,#161b22);border:1px solid #21262d;
     border-radius:12px;padding:1.8rem 2rem;margin-bottom:1.5rem;border-top:2px solid #238636">
  <h1 style="font-family:Space Mono,monospace;font-size:1.8rem;color:#e6edf3;margin:0">⚕️ FairAudit Platform</h1>
  <p style="color:#8b949e;margin:0.3rem 0 0.8rem;font-size:0.88rem">Healthcare AI — Upload Model · Train with Dataset · Run Fairness Audit</p>
  <span style="background:#1a4731;color:#56d364;border:1px solid #238636;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:600">OWASP ML Top 10</span>
  <span style="background:#1c2c4a;color:#58a6ff;border:1px solid #1f6feb;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:600;margin-left:6px">NIST AI RMF</span>
  <span style="background:#3d1f00;color:#ffa657;border:1px solid #d4800a;padding:3px 10px;border-radius:20px;font-size:11px;font-weight:600;margin-left:6px">MITRE ATLAS</span>
</div>""", unsafe_allow_html=True)

s = st.session_state.step
p1 = "✅" if s>1 else ("🔵" if s==1 else "⚪")
p2 = "✅" if s>2 else ("🔵" if s==2 else "⚪")
p3 = "✅" if st.session_state.audit_done else ("🔵" if s==3 else "⚪")
c1,c2,c3 = st.columns(3)
c1.markdown(f"""<div class="step-card {'active' if s==1 else 'done' if s>1 else ''}">
<div class="step-num">{p1} STEP 1</div><div class="step-title">Upload / Select Model</div>
<div class="step-desc">Choose sklearn model, HuggingFace, or upload .pkl file</div>
{'<div style="margin-top:0.6rem;font-family:Space Mono,monospace;font-size:0.72rem;color:#56d364">✓ ' + st.session_state.model_name + '</div>' if st.session_state.model_name else ''}
</div>""", unsafe_allow_html=True)
c2.markdown(f"""<div class="step-card {'active' if s==2 else 'done' if s>2 else ''}">
<div class="step-num">{p2} STEP 2</div><div class="step-title">Upload Dataset & Train</div>
<div class="step-desc">Upload Kaggle CSV and train the model with your configuration</div>
{'<div style="margin-top:0.6rem;font-family:Space Mono,monospace;font-size:0.72rem;color:#56d364">✓ Trained — ' + str(st.session_state.train_results.get("test_samples","")) + ' test samples</div>' if st.session_state.trained else ''}
</div>""", unsafe_allow_html=True)
c3.markdown(f"""<div class="step-card {'active' if s==3 else ''}">
<div class="step-num">{p3} STEP 3</div><div class="step-title">Run Fairness Audit</div>
<div class="step-desc">Fairlearn MetricFrame · DPD · EOD · Bias Mitigation · Charts</div>
{'<div style="margin-top:0.6rem;font-family:Space Mono,monospace;font-size:0.72rem;color:#56d364">✓ Audit complete</div>' if st.session_state.audit_done else ''}
</div>""", unsafe_allow_html=True)
st.markdown("---")

if st.session_state.step == 1:
    st.markdown('<div class="shdr"> Step 1 — Load Model</div>', unsafe_allow_html=True)
    model_source = st.radio("Where is your model?", [
        "Use a standard sklearn model (Gradient Boosting / Random Forest / etc.)",
        "Upload pre-trained model (.pkl file)",
        "Load from HuggingFace Hub"])

    if model_source == "Use a standard sklearn model (Gradient Boosting / Random Forest / etc.)":
        model_choice = st.selectbox("Select model", ["Gradient Boosting","Random Forest","Logistic Regression","Decision Tree"])
        st.markdown(f"""<div class="sbox"><span style="color:#56d364">✓</span> Selected: <b style="color:#e6edf3">{model_choice}</b><br>
        <span style="color:#56d364">✓</span> Source: scikit-learn (no download needed)<br>
        <span style="color:#56d364">✓</span> Same model used in your Jupyter session</div>""", unsafe_allow_html=True)
        if st.button("✅ Confirm Model — Go to Step 2"):
            from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
            from sklearn.linear_model import LogisticRegression
            from sklearn.tree import DecisionTreeClassifier
            mmap = {"Gradient Boosting":(GradientBoostingClassifier(n_estimators=100,random_state=42),"GradientBoostingClassifier"),
                    "Random Forest":(RandomForestClassifier(n_estimators=100,random_state=42),"RandomForestClassifier"),
                    "Logistic Regression":(LogisticRegression(max_iter=1000,random_state=42),"LogisticRegression"),
                    "Decision Tree":(DecisionTreeClassifier(max_depth=5,random_state=42),"DecisionTreeClassifier")}
            st.session_state.model_obj, st.session_state.model_name = mmap[model_choice]
            st.session_state.step = 2; st.rerun()

    elif model_source == "Upload pre-trained model (.pkl file)":
        st.markdown("""<div class="sbox"><span style="color:#58a6ff">ℹ</span> Upload a <b style="color:#e6edf3">.pkl</b> sklearn model file<br>
        <span style="color:#ffa657">⚠</span> Must be a scikit-learn compatible classifier</div>""", unsafe_allow_html=True)
        pkl_file = st.file_uploader("Upload .pkl model", type=["pkl"])
        if pkl_file:
            try:
                m = pickle.load(pkl_file)
                st.success(f"✅ Loaded: {type(m).__name__}")
                if st.button("✅ Confirm — Use This Model"):
                    st.session_state.model_obj = m; st.session_state.model_name = type(m).__name__
                    st.session_state.step = 2; st.rerun()
            except Exception as e:
                st.error(f"❌ Failed: {e}")

    elif model_source == "Load from HuggingFace Hub":
        st.markdown("""<div class="sbox"><span style="color:#58a6ff">ℹ</span> HuggingFace NLP models (BioBERT, ClinicalBERT) need text data<br>
        <span style="color:#ffa657">⚠</span> For tabular Kaggle CSV, sklearn models work best with Fairlearn<br>
        <span style="color:#56d364">✓</span> Select sklearn equivalent below</div>""", unsafe_allow_html=True)
        hf_choice = st.selectbox("Use sklearn equivalent", ["GradientBoostingClassifier (best for tabular)","RandomForestClassifier","LogisticRegression"])
        if st.button("✅ Load Model — Go to Step 2"):
            from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
            from sklearn.linear_model import LogisticRegression
            mmap2 = {"GradientBoostingClassifier (best for tabular)":(GradientBoostingClassifier(n_estimators=100,random_state=42),"GradientBoostingClassifier"),
                     "RandomForestClassifier":(RandomForestClassifier(n_estimators=100,random_state=42),"RandomForestClassifier"),
                     "LogisticRegression":(LogisticRegression(max_iter=1000,random_state=42),"LogisticRegression")}
            st.session_state.model_obj, st.session_state.model_name = mmap2[hf_choice]
            st.session_state.step = 2; st.rerun()

elif st.session_state.step == 2:
    st.markdown(f"""<div class="sbox" style="margin-bottom:1rem">
    <span style="color:#56d364">✓</span> Model ready: <b style="color:#e6edf3">{st.session_state.model_name}</b></div>""", unsafe_allow_html=True)
    st.markdown('<div class="shdr">📂 Step 2 — Upload Dataset & Train</div>', unsafe_allow_html=True)

    data_source = st.radio("Dataset source", ["Use Kaggle stroke dataset (already at ./data/)","Upload CSV from Kaggle"])
    df = None
    if data_source == "Use Kaggle stroke dataset (already at ./data/)":
        path = "./data/healthcare-dataset-stroke-data.csv"
        if os.path.exists(path):
            df = pd.read_csv(path)
            st.success(f"✅ Loaded stroke dataset — {len(df):,} rows, {df.shape[1]} columns")
        else:
            st.error("❌ Not found at ./data/ — please upload below")
    if data_source == "Upload CSV from Kaggle" or df is None:
        csv_file = st.file_uploader("📎 Upload Kaggle CSV dataset", type=["csv"])
        if csv_file:
            df = pd.read_csv(csv_file)
            st.success(f"✅ Loaded {csv_file.name} — {len(df):,} rows")

    if df is not None:
        with st.expander("👁 Preview Dataset", expanded=False):
            st.dataframe(df.head(15), use_container_width=True)
        st.markdown("### ⚙️ Training Configuration")
        ca,cb = st.columns(2)
        with ca:
            all_cols = [c for c in df.columns if c != "id"]
            default_t = "stroke" if "stroke" in df.columns else all_cols[-1]
            target_col = st.selectbox("🎯 Target column", all_cols, index=all_cols.index(default_t))
        with cb:
            sens_cols = [c for c in df.columns if c not in ["id",target_col]]
            default_s = "gender" if "gender" in df.columns else sens_cols[0]
            sensitive_col = st.selectbox(" Sensitive feature", sens_cols, index=sens_cols.index(default_s) if default_s in sens_cols else 0)
        cc,cd = st.columns(2)
        with cc: test_size = st.slider("Test split %", 10, 40, 20)
        with cd: balance_classes = st.toggle("Fix class imbalance", value=True)
        st.markdown(f"""<div class="sbox"><span style="color:#58a6ff">ℹ</span> Model: <b style="color:#e6edf3">{st.session_state.model_name}</b> | Target: <b style="color:#e6edf3">{target_col}</b> | Sensitive: <b style="color:#e6edf3">{sensitive_col}</b> | Test: <b style="color:#e6edf3">{test_size}%</b></div>""", unsafe_allow_html=True)

        if st.button(" Train Model Now"):
            try:
                from sklearn.preprocessing import LabelEncoder
                from sklearn.model_selection import train_test_split
                from sklearn.utils.class_weight import compute_class_weight
                from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
                import copy
                with st.spinner("Preprocessing & training..."):
                    df2 = df.copy().dropna()
                    if "id" in df2.columns: df2.drop(columns=["id"], inplace=True)
                    le = LabelEncoder()
                    for col in df2.select_dtypes(include="object").columns:
                        df2[col] = le.fit_transform(df2[col].astype(str))
                    sf = df2[sensitive_col]; X = df2.drop(columns=[target_col]); y = df2[target_col]
                    X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(X,y,sf,test_size=test_size/100,random_state=42)
                    model = copy.deepcopy(st.session_state.model_obj)
                    if balance_classes:
                        wts = compute_class_weight("balanced",classes=np.unique(y_train),y=y_train)
                        cw = {c:w for c,w in zip(np.unique(y_train),wts)}
                        model.fit(X_train,y_train,sample_weight=y_train.map(cw))
                    else:
                        model.fit(X_train,y_train)
                    y_pred = model.predict(X_test)
                    acc=accuracy_score(y_test,y_pred); prec=precision_score(y_test,y_pred,zero_division=0)
                    rec=recall_score(y_test,y_pred,zero_division=0); f1=f1_score(y_test,y_pred,zero_division=0)
                    st.session_state.train_results = dict(model=model,X_train=X_train,X_test=X_test,
                        y_train=y_train,y_test=y_test,sf_train=sf_train,sf_test=sf_test,y_pred=y_pred,
                        sensitive_col=sensitive_col,target_col=target_col,test_samples=len(y_test),
                        accuracy=acc,precision=prec,recall=rec,f1=f1)
                    st.session_state.trained = True
                st.success("✅ Training complete!")
                m1,m2,m3,m4 = st.columns(4)
                for col,(val,lbl,cl) in zip([m1,m2,m3,m4],[(f"{acc:.1%}","Accuracy","g" if acc>0.75 else "o"),
                    (f"{prec:.1%}","Precision","g" if prec>0.15 else "o"),(f"{rec:.1%}","Recall","g" if rec>0.5 else "o"),
                    (f"{f1:.1%}","F1 Score","g" if f1>0.3 else "o")]):
                    col.markdown(f'<div class="mcard"><div class="mval {cl}">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)
            except Exception as e:
                st.error(f"❌ Training failed: {e}")

    st.markdown("---")
    n1,_,n2 = st.columns([1,3,1])
    with n1:
        if st.button("← Back to Step 1"): st.session_state.step=1; st.rerun()
    with n2:
        if st.session_state.trained:
            if st.button("Next → Run Fairness Audit"): st.session_state.step=3; st.rerun()

elif st.session_state.step == 3:
    if not st.session_state.trained:
        st.error("❌ Complete Step 2 first.")
        if st.button("← Go to Step 2"): st.session_state.step=2; st.rerun()
        st.stop()
    tr = st.session_state.train_results; sc = tr["sensitive_col"]
    st.markdown(f"""<div class="sbox" style="margin-bottom:1rem">
    <span style="color:#56d364">✓</span> Model: <b style="color:#e6edf3">{st.session_state.model_name}</b> |
    Test samples: <b style="color:#e6edf3">{tr["test_samples"]}</b> |
    Sensitive: <b style="color:#58a6ff">{sc}</b> |
    Accuracy: <b style="color:#e6edf3">{tr["accuracy"]:.1%}</b></div>""", unsafe_allow_html=True)
    st.markdown('<div class="shdr"> Step 3 — Fairness Audit</div>', unsafe_allow_html=True)
    run_mitigation = st.toggle("Also run bias mitigation", value=True)

    if st.button("▶  Run Fairness Audit Now"):
        try:
            from fairlearn.metrics import (MetricFrame, demographic_parity_difference, equalized_odds_difference, selection_rate)
            from sklearn.metrics import accuracy_score, precision_score, recall_score
            with st.spinner("Running Fairlearn audit..."):
                metrics_dict = {"accuracy":accuracy_score,
                    "precision":lambda yt,yp:precision_score(yt,yp,zero_division=0),
                    "recall":lambda yt,yp:recall_score(yt,yp,zero_division=0),
                    "selection_rate":selection_rate}
                mf = MetricFrame(metrics=metrics_dict,y_true=tr["y_test"],y_pred=tr["y_pred"],sensitive_features=tr["sf_test"])
                dpd = demographic_parity_difference(tr["y_test"],tr["y_pred"],sensitive_features=tr["sf_test"])
                eod = equalized_odds_difference(tr["y_test"],tr["y_pred"],sensitive_features=tr["sf_test"])
            st.markdown("### 📊 Overall Fairness Metrics")
            ov = mf.overall
            def cs(v): return "g" if v<0.05 else ("o" if v<0.10 else "r")
            cols = st.columns(6)
            for col,(val,lbl,cl) in zip(cols,[
                (f'{ov["accuracy"]:.1%}',"Accuracy","g" if ov["accuracy"]>0.75 else "o"),
                (f'{ov["precision"]:.1%}',"Precision","g" if ov["precision"]>0.15 else "o"),
                (f'{ov["recall"]:.1%}',"Recall","g" if ov["recall"]>0.5 else "o"),
                (f'{ov["selection_rate"]:.1%}',"Selection Rate",""),
                (f'{abs(dpd):.4f}',"DPD ↓",cs(abs(dpd))),(f'{abs(eod):.4f}',"EOD ↓",cs(abs(eod)))]):
                col.markdown(f'<div class="mcard"><div class="mval {cl}">{val}</div><div class="mlbl">{lbl}</div></div>',unsafe_allow_html=True)
            fair=abs(dpd)<0.05 and abs(eod)<0.05; mild=abs(dpd)<0.10 and abs(eod)<0.10
            if fair: st.markdown('<div style="background:#1a4731;border:1px solid #238636;border-radius:10px;padding:1rem;color:#56d364;font-weight:600;text-align:center;margin:1rem 0">✅ FAIR — Bias scores within acceptable thresholds</div>',unsafe_allow_html=True)
            elif mild: st.markdown('<div style="background:#3d1f00;border:1px solid #d4800a;border-radius:10px;padding:1rem;color:#ffa657;font-weight:600;text-align:center;margin:1rem 0">⚠️ MILD BIAS — Disparity detected. Mitigation recommended.</div>',unsafe_allow_html=True)
            else: st.markdown('<div style="background:#3d0000;border:1px solid #f85149;border-radius:10px;padding:1rem;color:#f85149;font-weight:600;text-align:center;margin:1rem 0">🚨 SIGNIFICANT BIAS — Immediate mitigation required.</div>',unsafe_allow_html=True)

            st.markdown(f"### 📈 Metrics by `{sc}` Group")
            ct,cc = st.columns([1,1.6])
            with ct:
                st.markdown("**Group breakdown table**")
                st.dataframe(mf.by_group.style.format("{:.3f}").background_gradient(cmap="RdYlGn",axis=0),use_container_width=True)
                st.markdown(f"""<div class="sbox" style="margin-top:0.8rem">
                <span style="color:#58a6ff">DPD</span> {abs(dpd):.4f} — {"✅ Fair" if abs(dpd)<0.05 else "⚠️ Bias"}<br>
                <span style="color:#58a6ff">EOD</span> {abs(eod):.4f} — {"✅ Fair" if abs(eod)<0.05 else "⚠️ Bias"}<br>
                <span style="color:#8b949e">Fair threshold: &lt; 0.05</span></div>""",unsafe_allow_html=True)
            with cc:
                fig,axes = plt.subplots(1,4,figsize=(12,3.5)); fig.patch.set_facecolor("#161b22")
                for ax,(metric,color) in zip(axes,[("accuracy","#58a6ff"),("precision","#ffa657"),("recall","#56d364"),("selection_rate","#f85149")]):
                    groups=mf.by_group.index.astype(str); vals=mf.by_group[metric].values
                    bars=ax.bar(groups,vals,color=color,alpha=0.85); ax.set_facecolor("#0d1117")
                    ax.set_title(metric,color="#8b949e",fontsize=9,pad=6); ax.tick_params(colors="#8b949e",labelsize=8)
                    ax.set_xlabel(sc,color="#8b949e",fontsize=8)
                    for spine in ax.spines.values(): spine.set_visible(False)
                    for bar,val in zip(bars,vals):
                        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,f"{val:.3f}",ha="center",va="bottom",fontsize=8,color="#8b949e")
                fig.suptitle(f"{st.session_state.model_name} — Fairness by {sc}",color="#e6edf3",fontsize=10)
                plt.tight_layout(); st.pyplot(fig,use_container_width=True); plt.close()

            if run_mitigation:
                st.markdown("###  Bias Mitigation")
                with st.spinner("Running ExponentiatedGradient..."):
                    try:
                        from fairlearn.reductions import ExponentiatedGradient, DemographicParity
                        from sklearn.ensemble import GradientBoostingClassifier
                        mit = ExponentiatedGradient(estimator=GradientBoostingClassifier(n_estimators=50,random_state=42),constraints=DemographicParity())
                        mit.fit(tr["X_train"],tr["y_train"],sensitive_features=tr["sf_train"])
                        ypm=mit.predict(tr["X_test"])
                        dpd_post=demographic_parity_difference(tr["y_test"],ypm,sensitive_features=tr["sf_test"])
                        eod_post=equalized_odds_difference(tr["y_test"],ypm,sensitive_features=tr["sf_test"])
                        dr=(1-abs(dpd_post)/abs(dpd))*100 if abs(dpd)>0 else 0
                        er=(1-abs(eod_post)/abs(eod))*100 if abs(eod)>0 else 0
                        m1,m2,m3,m4=st.columns(4)
                        m1.markdown(f'<div class="mcard"><div class="mval o">{abs(dpd):.4f}</div><div class="mlbl">DPD Before</div></div>',unsafe_allow_html=True)
                        m2.markdown(f'<div class="mcard"><div class="mval g">{abs(dpd_post):.4f}</div><div class="mlbl">DPD After</div></div>',unsafe_allow_html=True)
                        m3.markdown(f'<div class="mcard"><div class="mval o">{abs(eod):.4f}</div><div class="mlbl">EOD Before</div></div>',unsafe_allow_html=True)
                        m4.markdown(f'<div class="mcard"><div class="mval g">{abs(eod_post):.4f}</div><div class="mlbl">EOD After</div></div>',unsafe_allow_html=True)
                        st.markdown(f"""<div class="sbox" style="margin-top:1rem">
                        <span style="color:#56d364">✓</span> DPD reduced by <b style="color:#56d364">{dr:.1f}%</b> ({abs(dpd):.4f} → {abs(dpd_post):.4f})<br>
                        <span style="color:#56d364">✓</span> EOD reduced by <b style="color:#56d364">{er:.1f}%</b> ({abs(eod):.4f} → {abs(eod_post):.4f})<br>
                        <span style="color:#58a6ff">ℹ</span> Algorithm: ExponentiatedGradient + DemographicParity constraint</div>""",unsafe_allow_html=True)
                        fig2,ax2=plt.subplots(figsize=(6,3.5)); fig2.patch.set_facecolor("#161b22"); ax2.set_facecolor("#0d1117")
                        x=np.arange(2); w=0.3
                        ax2.bar(x-w/2,[abs(dpd),abs(eod)],w,label="Before",color="#ffa657",alpha=0.85)
                        ax2.bar(x+w/2,[abs(dpd_post),abs(eod_post)],w,label="After",color="#56d364",alpha=0.85)
                        ax2.set_xticks(x); ax2.set_xticklabels(["DPD","EOD"],color="#8b949e",fontsize=11)
                        ax2.tick_params(colors="#8b949e"); ax2.axhline(0.05,color="#f85149",linestyle="--",alpha=0.6,linewidth=1.2)
                        ax2.text(1.65,0.052,"Fair threshold",color="#f85149",fontsize=9)
                        ax2.legend(facecolor="#161b22",labelcolor="#e6edf3",fontsize=9)
                        ax2.set_title("Before vs After Mitigation",color="#e6edf3",fontsize=11)
                        for spine in ax2.spines.values(): spine.set_visible(False)
                        plt.tight_layout()
                        col_c,_=st.columns([1,1])
                        with col_c: st.pyplot(fig2,use_container_width=True)
                        plt.close()
                    except Exception as e:
                        st.warning(f"Mitigation error: {e}")
            st.session_state.audit_done = True
            st.success(" Full audit complete!")
        except Exception as e:
            st.error(f"❌ Audit failed: {e}")

    st.markdown("---")
    n1,_,n2=st.columns([1,3,1])
    with n1:
        if st.button("← Back to Step 2"): st.session_state.step=2; st.rerun()
    with n2:
        if st.button("🔄 Start Over"):
            for k in list(st.session_state.keys()): del st.session_state[k]
            st.rerun()

st.markdown('<div style="margin-top:3rem;padding:1rem;border-top:1px solid #21262d;text-align:center;font-family:Space Mono,monospace;font-size:11px;color:#8b949e;letter-spacing:2px">FAIRAUDIT PLATFORM · FAIRLEARN + SCIKIT-LEARN · OWASP ML TOP 10</div>',unsafe_allow_html=True)
'''

with open("fairlearn_ui.py", "w", encoding="utf-8") as f:
    f.write(code)

print("✅ fairlearn_ui.py written successfully!")
print("📁 Location:", __import__('os').path.abspath("fairlearn_ui.py"))
print("\n✅ Now run in PowerShell:")
print('   streamlit run fairlearn_ui.py')

In [ ]:
# Fix - replace "Healthcare AI" with generic text
with open("fairlearn_ui.py", "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace(
    "Healthcare AI — Upload Model · Train with Dataset · Run Fairness Audit",
    "AI Fairness Platform — Upload Model · Train with Dataset · Run Fairness Audit"
)

content = content.replace(
    "Upload Kaggle CSV and train the model with your configuration",
    "Upload any CSV dataset and train the model with your configuration"
)

content = content.replace(
    "Upload Kaggle CSV dataset",
    "Upload any CSV dataset"
)

content = content.replace(
    "Use Kaggle stroke dataset (already at ./data/)",
    "Use stroke dataset (already at ./data/)"
)

with open("fairlearn_ui.py", "w", encoding="utf-8") as f:
    f.write(content)

print("✅ Fixed! Now refresh your browser at localhost:8501")

In [ ]:
import pickle, os

# Save model for UI demo
with open("trained_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save test data for UI demo
test_df = X_test.copy()
test_df['stroke'] = y_test.values
test_df['gender'] = sf_test.values
test_df.to_csv("test_data.csv", index=False)

print("✅ Both files saved and ready for demo!")
print("trained_model.pkl →", os.path.abspath("trained_model.pkl"))
print("test_data.csv     →", os.path.abspath("test_data.csv"))

In [ ]:
import pickle, os

# Save trained model
with open("trained_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save test data
test_df = X_test.copy()
test_df['stroke'] = y_test.values
test_df['gender'] = sf_test.values
test_df.to_csv("test_data.csv", index=False)

print("✅ trained_model.pkl saved!")
print("✅ test_data.csv saved!")
print("📁 Folder:", os.path.abspath("."))

In [3]:
# Read the current file
with open("fairlearn_ui.py", "r", encoding="utf-8") as f:
    content = f.read()

# Replace title text
content = content.replace(
    "FairAudit — Fairlearn Tool",
    "AI Model Fairness Audit Platform"
)
content = content.replace(
    "⚖️ FairAudit",
    "⚖️ AI Model Fairness Audit Platform"
)
content = content.replace(
    "AI Model Fairness Audit Tool — Works with ANY trained model and ANY dataset",
    "Upload Any Trained Model · Test Data · Run Fairlearn Bias Audit"
)

# Save the updated file
with open("fairlearn_ui.py", "w", encoding="utf-8") as f:
    f.write(content)

print("✅ Title updated!")
print("Now restart Streamlit — press Ctrl+C in PowerShell then run:")
print("streamlit run fairlearn_ui.py")

✅ Title updated!
Now restart Streamlit — press Ctrl+C in PowerShell then run:
streamlit run fairlearn_ui.py


In [2]:
# Install datasets library
import subprocess
subprocess.run(["pip", "install", "datasets"], capture_output=True)
print("✅ Installed!")

✅ Installed!


In [3]:
from datasets import load_dataset
import pandas as pd

# Load German Credit dataset directly from HuggingFace
dataset = load_dataset(
    "AiresPucrs/german-credit-data",
    split="train"
)

# Convert to pandas
df = dataset.to_pandas()

print("✅ Dataset loaded from HuggingFace!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(5))
print(df.dtypes)

ModuleNotFoundError: No module named 'datasets'

In [4]:
from datasets import load_dataset
import pandas as pd

# Load German Credit dataset directly from HuggingFace
dataset = load_dataset(
    "AiresPucrs/german-credit-data",
    split="train"
)

# Convert to pandas
df = dataset.to_pandas()

print("✅ Dataset loaded from HuggingFace!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(5))
print(df.dtypes)

C:\Users\nikhil.malige.RHYMTECH\.conda\envs\fairlearn-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\nikhil.malige.RHYMTECH\.conda\envs\fairlearn-env\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nikhil.malige.RHYMTECH\.cache\huggingface\hub\datasets--AiresPucrs--german-credit-data. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develop

✅ Dataset loaded from HuggingFace!
Shape: (1000, 10)
Columns: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own            None           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little             None           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  Risk  
0         6             radio/TV  good  
1        48             radio/TV   bad  
2        12            education  good  
3        42  furniture/equipment  good  
4        24                  car   bad  
Age                  int64
Sex                 object
Job                  int64
Housing         

In [5]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ── Clean Data ────────────────────────────────────────────
df = df.copy()
df.dropna(inplace=True)

# ── Encode categorical columns ────────────────────────────
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

print("✅ Encoding done")
print("Columns:", df.columns.tolist())
print("Risk values:", df['Risk'].unique())
print("Sex values:",  df['Sex'].unique())
print("Age range:",   df['Age'].min(), "-", df['Age'].max())

# ── Create Age Group as Sensitive Feature ─────────────────
# 0 = Young  (age < 30)
# 1 = Middle (age 30-50)
# 2 = Senior (age > 50)
df['age_group'] = pd.cut(
    df['Age'],
    bins=[0, 30, 50, 100],
    labels=[0, 1, 2]
).astype(int)

# ── Split ─────────────────────────────────────────────────
X  = df.drop(columns=['Risk'])
y  = df['Risk']           # 0 = Bad credit, 1 = Good credit
sf = df['Sex']            # 0 = Female, 1 = Male

X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(
    X, y, sf, test_size=0.2, random_state=42
)

print(f"\nTrain rows: {len(X_train)}")
print(f"Test rows:  {len(X_test)}")

# ── Train Model ───────────────────────────────────────────
print("\nTraining RandomForest model...")
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score
acc = accuracy_score(y_test, model.predict(X_test))
print(f"✅ Training complete! Accuracy: {acc:.1%}")

# ── Save Model ────────────────────────────────────────────
with open("german_credit_model.pkl", "wb") as f:
    pickle.dump(model, f)

# ── Save Test Data ────────────────────────────────────────
test_df = X_test.copy()
test_df['Risk'] = y_test.values
test_df['Sex']  = sf_test.values
test_df.to_csv("german_credit_test.csv", index=False)

print("\n✅ german_credit_model.pkl saved!")
print("✅ german_credit_test.csv saved!")
print("📁 Location:", os.path.abspath("."))
print("📊 Test rows:", len(test_df))
print("📋 Columns:", test_df.columns.tolist())

✅ Encoding done
Columns: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'Risk']
Risk values: [0 1]
Sex values: [0 1]
Age range: 19 - 75

Train rows: 417
Test rows:  105

Training RandomForest model...
✅ Training complete! Accuracy: 59.0%

✅ german_credit_model.pkl saved!
✅ german_credit_test.csv saved!
📁 Location: C:\Users\nikhil.malige.RHYMTECH\anaconda_projects\9ff078f8-62c7-40b1-b5ef-6084b1f0b4ba
📊 Test rows: 105
📋 Columns: ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration', 'Purpose', 'age_group', 'Risk']


In [6]:
import os

m = os.path.exists("german_credit_model.pkl")
d = os.path.exists("german_credit_test.csv")

print("german_credit_model.pkl →", "✅ READY" if m else "❌ NOT FOUND")
print("german_credit_test.csv  →", "✅ READY" if d else "❌ NOT FOUND")
print("\n📌 Now open UI → upload both files")

german_credit_model.pkl → ✅ READY
german_credit_test.csv  → ✅ READY

📌 Now open UI → upload both files


In [7]:
import os, pickle, pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

os.environ['KAGGLE_USERNAME'] = 'nikhilmalige'
os.environ['KAGGLE_KEY']      = 'fdd621db4c083700ebe82782a3648dc8'

import kaggle
kaggle.api.authenticate()
print("✅ Kaggle ready!")

def save_model_and_test(model, X_test, y_test, sf_test,
                         target_col, sensitive_col,
                         model_name, csv_name):
    with open(model_name, "wb") as f:
        pickle.dump(model, f)
    test_df = X_test.copy()
    test_df[target_col]    = y_test.values
    test_df[sensitive_col] = sf_test.values
    test_df.to_csv(csv_name, index=False)
    acc = (model.predict(X_test) == y_test).mean()
    print(f"✅ {model_name} saved | Accuracy: {acc:.1%} | Test rows: {len(test_df)}")

✅ Kaggle ready!


In [8]:
# Download
kaggle.api.dataset_download_files(
    'pavansubhasht/ibm-hr-analytics-attrition-dataset',
    path='./data/hr',
    unzip=True
)

# Load
df = pd.read_csv('./data/hr/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df.dropna(inplace=True)

# Encode
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

# Split
X  = df.drop(columns=['Attrition','EmployeeNumber'])
y  = df['Attrition']    # 0=No, 1=Yes
sf = df['Gender']       # 0=Female, 1=Male

X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(
    X, y, sf, test_size=0.2, random_state=42
)

# Train
print("Training HR Attrition model...")
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save
save_model_and_test(model, X_test, y_test, sf_test,
                    'Attrition', 'Gender',
                    'hr_model.pkl',
                    'hr_test.csv')

Dataset URL: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
Training HR Attrition model...
✅ hr_model.pkl saved | Accuracy: 89.1% | Test rows: 294


In [9]:
# Download
kaggle.api.dataset_download_files(
    'brycecf/give-me-some-credit-dataset',
    path='./data/credit',
    unzip=True
)

# Load
df = pd.read_csv('./data/credit/cs-training.csv')
df.dropna(inplace=True)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# Age group sensitive feature
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 50, 100],
    labels=[0, 1, 2]
).astype(int)

print("Columns:", df.columns.tolist())

# Split
X  = df.drop(columns=['SeriousDlqin2yrs'])
y  = df['SeriousDlqin2yrs']   # 0=No default, 1=Default
sf = df['age_group']

X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(
    X, y, sf, test_size=0.2, random_state=42
)

# Balance
wts = compute_class_weight('balanced',classes=np.unique(y_train),y=y_train)
cw  = dict(zip(np.unique(y_train), wts))

# Train
print("Training Give Me Credit model (250K rows)...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train, sample_weight=y_train.map(cw))

# Save
save_model_and_test(model, X_test, y_test, sf_test,
                    'SeriousDlqin2yrs', 'age_group',
                    'credit_model.pkl',
                    'credit_test.csv')

Dataset URL: https://www.kaggle.com/datasets/brycecf/give-me-some-credit-dataset


ValueError: Cannot convert float NaN to integer

In [10]:
# Download
kaggle.api.dataset_download_files(
    'henriqueyamahata/bank-marketing',
    path='./data/bank',
    unzip=True
)

# Load
df = pd.read_csv('./data/bank/bank-additional-full.csv', sep=';')
df.dropna(inplace=True)

# Encode
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

# Create age group
df['age_group'] = (df['age'] >= 40).astype(int)

# Split
X  = df.drop(columns=['y'])
y  = df['y']            # 0=No deposit, 1=Yes deposit
sf = df['age_group']    # 0=Under 40, 1=Over 40

X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(
    X, y, sf, test_size=0.2, random_state=42
)

# Train
print("Training Bank Marketing model (45K rows)...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save
save_model_and_test(model, X_test, y_test, sf_test,
                    'y', 'age_group',
                    'bank_model.pkl',
                    'bank_test.csv')

Dataset URL: https://www.kaggle.com/datasets/henriqueyamahata/bank-marketing
Training Bank Marketing model (45K rows)...
✅ bank_model.pkl saved | Accuracy: 91.5% | Test rows: 8238


In [12]:
with open("fairlearn_ui.py", "r", encoding="utf-8") as f:
    content = f.read()

# Fix 1 — Use better split ratio for mitigation
content = content.replace(
    "Xtr,Xte,ytr,yte,sftr,sfte = train_test_split(Xm,ym,sfm,test_size=0.3,random_state=42)",
    "Xtr,Xte,ytr,yte,sftr,sfte = train_test_split(Xm,ym,sfm,test_size=0.2,random_state=42)"
)

# Fix 2 — Use EqualizedOdds instead of DemographicParity
content = content.replace(
    "constraints=DemographicParity()",
    "constraints=EqualizedOdds()"
)
content = content.replace(
    "from fairlearn.reductions import ExponentiatedGradient, DemographicParity",
    "from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds"
)

with open("fairlearn_ui.py", "w", encoding="utf-8") as f:
    f.write(content)

print("✅ Fixed!")

✅ Fixed!


In [13]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# ═══════════════════════════════════════
# MODEL 1 — BANK MARKETING (45K rows)
# ═══════════════════════════════════════
print("="*50)
print("MODEL 1 — Bank Marketing")
print("="*50)

df = pd.read_csv('./data/bank/bank-additional-full.csv', sep=';')
df.dropna(inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df['age_group'] = (df['age'] >= 40).astype(int)

X  = df.drop(columns=['y'])
y  = df['y']
sf = df['age_group']

X_train,X_test,y_train,y_test,sf_train,sf_test = train_test_split(
    X, y, sf, test_size=0.2, random_state=42)

model1 = RandomForestClassifier(n_estimators=100, random_state=42)
model1.fit(X_train, y_train)

with open("bank_model.pkl","wb") as f: pickle.dump(model1, f)
test_df = X_test.copy(); test_df['y'] = y_test.values
test_df.to_csv("bank_test.csv", index=False)

acc = (model1.predict(X_test) == y_test).mean()
print(f"✅ Accuracy: {acc:.1%}")
print(f"✅ bank_model.pkl saved")
print(f"✅ bank_test.csv saved")
print(f"   Target: y | Sensitive: age_group | Rows: {len(test_df)}")


# ═══════════════════════════════════════
# MODEL 2 — GIVE ME CREDIT (250K rows)
# ═══════════════════════════════════════
print("\n" + "="*50)
print("MODEL 2 — Give Me Credit")
print("="*50)

df2 = pd.read_csv('./data/credit/cs-training.csv')
df2.dropna(inplace=True)
if 'Unnamed: 0' in df2.columns:
    df2.drop(columns=['Unnamed: 0'], inplace=True)

df2['age_group'] = pd.cut(
    df2['age'],
    bins=[0, 30, 50, 100],
    labels=[0, 1, 2]
).astype(int)

X2  = df2.drop(columns=['SeriousDlqin2yrs'])
y2  = df2['SeriousDlqin2yrs']
sf2 = df2['age_group']

X_train2,X_test2,y_train2,y_test2,sf_train2,sf_test2 = train_test_split(
    X2, y2, sf2, test_size=0.2, random_state=42)

wts2 = compute_class_weight('balanced',
       classes=np.unique(y_train2), y=y_train2)
cw2  = dict(zip(np.unique(y_train2), wts2))

print("Training... (may take 2-3 mins)")
model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train2, y_train2, sample_weight=y_train2.map(cw2))

with open("credit_model.pkl","wb") as f: pickle.dump(model2, f)
test_df2 = X_test2.copy(); test_df2['SeriousDlqin2yrs'] = y_test2.values
test_df2.to_csv("credit_test.csv", index=False)

acc2 = (model2.predict(X_test2) == y_test2).mean()
print(f"✅ Accuracy: {acc2:.1%}")
print(f"✅ credit_model.pkl saved")
print(f"✅ credit_test.csv saved")
print(f"   Target: SeriousDlqin2yrs | Sensitive: age_group | Rows: {len(test_df2)}")


# ═══════════════════════════════════════
# MODEL 3 — IBM HR ATTRITION
# ═══════════════════════════════════════
print("\n" + "="*50)
print("MODEL 3 — IBM HR Attrition")
print("="*50)

df3 = pd.read_csv('./data/hr/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df3.dropna(inplace=True)

le3 = LabelEncoder()
for col in df3.select_dtypes(include='object').columns:
    df3[col] = le3.fit_transform(df3[col].astype(str))

X3  = df3.drop(columns=['Attrition','EmployeeNumber'])
y3  = df3['Attrition']
sf3 = df3['Gender']

X_train3,X_test3,y_train3,y_test3,sf_train3,sf_test3 = train_test_split(
    X3, y3, sf3, test_size=0.2, random_state=42)

model3 = GradientBoostingClassifier(n_estimators=100, random_state=42)
model3.fit(X_train3, y_train3)

with open("hr_model.pkl","wb") as f: pickle.dump(model3, f)
test_df3 = X_test3.copy(); test_df3['Attrition'] = y_test3.values
test_df3.to_csv("hr_test.csv", index=False)

acc3 = (model3.predict(X_test3) == y_test3).mean()
print(f"✅ Accuracy: {acc3:.1%}")
print(f"✅ hr_model.pkl saved")
print(f"✅ hr_test.csv saved")
print(f"   Target: Attrition | Sensitive: Gender | Rows: {len(test_df3)}")


# ═══════════════════════════════════════
# MODEL 4 — ADULT CENSUS INCOME
# ═══════════════════════════════════════
print("\n" + "="*50)
print("MODEL 4 — Adult Census Income")
print("="*50)

try:
    df4 = pd.read_csv('./data/adult/adult.csv')
    df4.columns = df4.columns.str.strip()
    df4 = df4.replace(' ?', np.nan).dropna()

    le4 = LabelEncoder()
    for col in df4.select_dtypes(include='object').columns:
        df4[col] = le4.fit_transform(df4[col].astype(str))

    X4  = df4.drop(columns=['income'])
    y4  = df4['income']
    sf4 = df4['sex']

    X_train4,X_test4,y_train4,y_test4,sf_train4,sf_test4 = train_test_split(
        X4, y4, sf4, test_size=0.2, random_state=42)

    model4 = RandomForestClassifier(n_estimators=100, random_state=42)
    model4.fit(X_train4, y_train4)

    with open("adult_model.pkl","wb") as f: pickle.dump(model4, f)
    test_df4 = X_test4.copy(); test_df4['income'] = y_test4.values
    test_df4.to_csv("adult_test.csv", index=False)

    acc4 = (model4.predict(X_test4) == y_test4).mean()
    print(f"✅ Accuracy: {acc4:.1%}")
    print(f"✅ adult_model.pkl saved")
    print(f"✅ adult_test.csv saved")
    print(f"   Target: income | Sensitive: sex | Rows: {len(test_df4)}")
except Exception as e:
    print(f"⚠️ Adult skipped: {e}")


# ═══════════════════════════════════════
# FINAL STATUS CHECK
# ═══════════════════════════════════════
print("\n" + "="*50)
print("ALL FILES STATUS")
print("="*50)

files = [
    ("bank_model.pkl",   "bank_test.csv",   "y",                "age_group"),
    ("credit_model.pkl", "credit_test.csv", "SeriousDlqin2yrs", "age_group"),
    ("hr_model.pkl",     "hr_test.csv",     "Attrition",        "Gender"),
    ("adult_model.pkl",  "adult_test.csv",  "income",           "sex"),
]

print(f"{'Model File':<25} {'Test CSV':<25} {'Target':<22} {'Sensitive'}")
print("-"*90)
for m,t,tc,sc in files:
    ms = "✅" if os.path.exists(m) else "❌"
    ts = "✅" if os.path.exists(t) else "❌"
    print(f"{ms} {m:<23} {ts} {t:<23} {tc:<20} {sc}")

print("\n📌 Upload guide:")
print("-"*50)
for m,t,tc,sc in files:
    if os.path.exists(m) and os.path.exists(t):
        print(f"Model: {m}")
        print(f"CSV:   {t}")
        print(f"       Target={tc} | Sensitive={sc}")
        print()

MODEL 1 — Bank Marketing
✅ Accuracy: 91.5%
✅ bank_model.pkl saved
✅ bank_test.csv saved
   Target: y | Sensitive: age_group | Rows: 8238

MODEL 2 — Give Me Credit


ValueError: Cannot convert float NaN to integer

In [14]:
# ═══════════════════════════════════════
# MODEL 2 — GIVE ME CREDIT (250K rows) FIXED
# ═══════════════════════════════════════
print("="*50)
print("MODEL 2 — Give Me Credit (Fixed)")
print("="*50)

df2 = pd.read_csv('./data/credit/cs-training.csv')

# Drop unnamed column
if 'Unnamed: 0' in df2.columns:
    df2.drop(columns=['Unnamed: 0'], inplace=True)

# Check age column
print("Age nulls before:", df2['age'].isnull().sum())
print("Age range:", df2['age'].min(), "-", df2['age'].max())

# Fix 1 — Drop rows where age is null
df2.dropna(subset=['age', 'SeriousDlqin2yrs'], inplace=True)

# Fix 2 — Fill remaining nulls with median
df2.fillna(df2.median(numeric_only=True), inplace=True)

print("Age nulls after:", df2['age'].isnull().sum())
print("Shape after cleaning:", df2.shape)

# Fix 3 — Create age group safely
df2['age_group'] = 0  # default
df2.loc[df2['age'] < 30, 'age_group']  = 0   # Young
df2.loc[(df2['age'] >= 30) & (df2['age'] < 50), 'age_group'] = 1  # Middle
df2.loc[df2['age'] >= 50, 'age_group'] = 2   # Senior

df2['age_group'] = df2['age_group'].astype(int)

print("age_group values:", df2['age_group'].value_counts().to_dict())

# Split
X2  = df2.drop(columns=['SeriousDlqin2yrs'])
y2  = df2['SeriousDlqin2yrs'].astype(int)
sf2 = df2['age_group']

X_train2,X_test2,y_train2,y_test2,sf_train2,sf_test2 = train_test_split(
    X2, y2, sf2, test_size=0.2, random_state=42)

# Balance classes
wts2 = compute_class_weight('balanced',
       classes=np.unique(y_train2), y=y_train2)
cw2  = dict(zip(np.unique(y_train2), wts2))

# Train
print("Training... (2-3 mins for 250K rows)")
model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train2, y_train2, sample_weight=y_train2.map(cw2))

# Save
with open("credit_model.pkl","wb") as f:
    pickle.dump(model2, f)

test_df2 = X_test2.copy()
test_df2['SeriousDlqin2yrs'] = y_test2.values
test_df2.to_csv("credit_test.csv", index=False)

acc2 = (model2.predict(X_test2) == y_test2).mean()
print(f"✅ Accuracy: {acc2:.1%}")
print(f"✅ credit_model.pkl saved")
print(f"✅ credit_test.csv saved")
print(f"   Target: SeriousDlqin2yrs | Sensitive: age_group | Rows: {len(test_df2)}")

MODEL 2 — Give Me Credit (Fixed)
Age nulls before: 0
Age range: 0 - 109
Age nulls after: 0
Shape after cleaning: (150000, 11)
age_group values: {2: 83619, 1: 57560, 0: 8821}
Training... (2-3 mins for 250K rows)
✅ Accuracy: 93.7%
✅ credit_model.pkl saved
✅ credit_test.csv saved
   Target: SeriousDlqin2yrs | Sensitive: age_group | Rows: 30000


In [15]:
# ═══════════════════════════════════════
# MODEL 3 — IBM HR (run after Model 2)
# ═══════════════════════════════════════
print("\n" + "="*50)
print("MODEL 3 — IBM HR Attrition")
print("="*50)

df3 = pd.read_csv('./data/hr/WA_Fn-UseC_-HR-Employee-Attrition.csv')
df3.dropna(inplace=True)

le3 = LabelEncoder()
for col in df3.select_dtypes(include='object').columns:
    df3[col] = le3.fit_transform(df3[col].astype(str))

X3  = df3.drop(columns=['Attrition','EmployeeNumber'])
y3  = df3['Attrition']
sf3 = df3['Gender']

X_train3,X_test3,y_train3,y_test3,sf_train3,sf_test3 = train_test_split(
    X3, y3, sf3, test_size=0.2, random_state=42)

model3 = GradientBoostingClassifier(n_estimators=100, random_state=42)
model3.fit(X_train3, y_train3)

with open("hr_model.pkl","wb") as f: pickle.dump(model3, f)
test_df3 = X_test3.copy()
test_df3['Attrition'] = y_test3.values
test_df3.to_csv("hr_test.csv", index=False)

acc3 = (model3.predict(X_test3) == y_test3).mean()
print(f"✅ Accuracy: {acc3:.1%}")
print(f"✅ hr_model.pkl saved")
print(f"✅ hr_test.csv saved")
print(f"   Target: Attrition | Sensitive: Gender | Rows: {len(test_df3)}")


# ═══════════════════════════════════════
# MODEL 4 — ADULT CENSUS
# ═══════════════════════════════════════
print("\n" + "="*50)
print("MODEL 4 — Adult Census Income")
print("="*50)

try:
    df4 = pd.read_csv('./data/adult/adult.csv')
    df4.columns = df4.columns.str.strip()
    df4 = df4.replace(' ?', np.nan).dropna()

    le4 = LabelEncoder()
    for col in df4.select_dtypes(include='object').columns:
        df4[col] = le4.fit_transform(df4[col].astype(str))

    X4  = df4.drop(columns=['income'])
    y4  = df4['income']
    sf4 = df4['sex']

    X_train4,X_test4,y_train4,y_test4,sf_train4,sf_test4 = train_test_split(
        X4, y4, sf4, test_size=0.2, random_state=42)

    model4 = RandomForestClassifier(n_estimators=100, random_state=42)
    model4.fit(X_train4, y_train4)

    with open("adult_model.pkl","wb") as f: pickle.dump(model4, f)
    test_df4 = X_test4.copy()
    test_df4['income'] = y_test4.values
    test_df4.to_csv("adult_test.csv", index=False)

    acc4 = (model4.predict(X_test4) == y_test4).mean()
    print(f"✅ Accuracy: {acc4:.1%}")
    print(f"✅ adult_model.pkl saved")
    print(f"✅ adult_test.csv saved")
    print(f"   Target: income | Sensitive: sex | Rows: {len(test_df4)}")

except Exception as e:
    print(f"⚠️ Adult skipped: {e}")


# ═══════════════════════════════════════
# FINAL STATUS
# ═══════════════════════════════════════
print("\n" + "="*50)
print("ALL FILES STATUS")
print("="*50)

files = [
    ("bank_model.pkl",   "bank_test.csv",   "y",                "age_group"),
    ("credit_model.pkl", "credit_test.csv", "SeriousDlqin2yrs", "age_group"),
    ("hr_model.pkl",     "hr_test.csv",     "Attrition",        "Gender"),
    ("adult_model.pkl",  "adult_test.csv",  "income",           "sex"),
]

for m,t,tc,sc in files:
    ms = "✅" if os.path.exists(m) else "❌"
    ts = "✅" if os.path.exists(t) else "❌"
    print(f"{ms} {m:<25} {ts} {t:<25} Target={tc} | Sensitive={sc}")


MODEL 3 — IBM HR Attrition
✅ Accuracy: 89.1%
✅ hr_model.pkl saved
✅ hr_test.csv saved
   Target: Attrition | Sensitive: Gender | Rows: 294

MODEL 4 — Adult Census Income
⚠️ Adult skipped: [Errno 2] No such file or directory: './data/adult/adult.csv'

ALL FILES STATUS
✅ bank_model.pkl            ✅ bank_test.csv             Target=y | Sensitive=age_group
✅ credit_model.pkl          ✅ credit_test.csv           Target=SeriousDlqin2yrs | Sensitive=age_group
✅ hr_model.pkl              ✅ hr_test.csv               Target=Attrition | Sensitive=Gender
❌ adult_model.pkl           ❌ adult_test.csv            Target=income | Sensitive=sex


In [16]:
# Verify the NEW bank model is saved correctly
import pickle
import pandas as pd

# Load new model and check training features
with open("bank_model.pkl", "rb") as f:
    m = pickle.load(f)

print("Model type:", type(m).__name__)
print("Training features:", m.feature_names_in_.tolist())
print("Number of features:", len(m.feature_names_in_))

Model type: RandomForestClassifier
Training features: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'age_group']
Number of features: 21


In [17]:
import pickle
import pandas as pd

# Check model
with open("bank_model.pkl", "rb") as f:
    m = pickle.load(f)

print("=== MODEL FEATURES ===")
features = m.feature_names_in_.tolist()
print(features)
print("Total features:", len(features))
print()

# Check test data
df = pd.read_csv("bank_test.csv")
print("=== TEST CSV COLUMNS ===")
print(df.columns.tolist())
print("Total columns:", len(df.columns))
print()

# Check match
model_cols = set(features)
csv_cols   = set(df.columns)

missing = model_cols - csv_cols
extra   = csv_cols - model_cols

print("=== STATUS ===")
print("Missing in CSV:", missing if missing else "✅ None")
print("Extra in CSV:  ", extra   if extra   else "✅ None")

if not missing:
    print("\n✅ PERFECT MATCH — Ready to upload!")
    print("\nIn FairAudit UI select:")
    print("Target    → y")
    print("Sensitive → age_group")
else:
    print("\n❌ MISMATCH — Need to retrain")

=== MODEL FEATURES ===
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'age_group']
Total features: 21

=== TEST CSV COLUMNS ===
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'age_group', 'y']
Total columns: 22

=== STATUS ===
Missing in CSV: ✅ None
Extra in CSV:   {'y'}

✅ PERFECT MATCH — Ready to upload!

In FairAudit UI select:
Target    → y
Sensitive → age_group


In [18]:
import pandas as pd, numpy as np, pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

df2 = pd.read_csv('./data/credit/cs-training.csv')
if 'Unnamed: 0' in df2.columns:
    df2.drop(columns=['Unnamed: 0'], inplace=True)

# Clean nulls
df2.dropna(subset=['age', 'SeriousDlqin2yrs'], inplace=True)
df2.fillna(df2.median(numeric_only=True), inplace=True)

# ✅ BINARY age_group — this is what mitigation needs
df2['age_group'] = (df2['age'] >= 40).astype(int)
# 0 = Under 40 (young), 1 = Over 40 (senior)

print("age_group distribution:")
print(df2['age_group'].value_counts())
print("age_group unique values:", df2['age_group'].nunique())  # Must print 2

X2  = df2.drop(columns=['SeriousDlqin2yrs'])
y2  = df2['SeriousDlqin2yrs'].astype(int)
sf2 = df2['age_group']

X_train2, X_test2, y_train2, y_test2, sf_train2, sf_test2 = train_test_split(
    X2, y2, sf2, test_size=0.2, random_state=42)

wts2 = compute_class_weight('balanced', classes=np.unique(y_train2), y=y_train2)
cw2  = dict(zip(np.unique(y_train2), wts2))

print("Training...")
model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X_train2, y_train2, sample_weight=y_train2.map(cw2))

with open("credit_model.pkl", "wb") as f:
    pickle.dump(model2, f)

test_df2 = X_test2.copy()
test_df2['SeriousDlqin2yrs'] = y_test2.values
test_df2.to_csv("credit_test.csv", index=False)

# Preview bias BEFORE upload
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
y_pred = model2.predict(X_test2)
dpd = demographic_parity_difference(y_test2, y_pred, sensitive_features=sf_test2)
eod = equalized_odds_difference(y_test2, y_pred, sensitive_features=sf_test2)

print(f"\n✅ Saved: credit_model.pkl + credit_test.csv")
print(f"DPD: {abs(dpd):.4f}  {'⚠️ Bias' if abs(dpd)>0.05 else '✅ Fair'}")
print(f"EOD: {abs(eod):.4f}  {'⚠️ Bias' if abs(eod)>0.05 else '✅ Fair'}")
print(f"\nIn FairAudit UI:")
print(f"  Target    → SeriousDlqin2yrs")
print(f"  Sensitive → age_group  ← NOT age")

age_group distribution:
age_group
1    117996
0     32004
Name: count, dtype: int64
age_group unique values: 2
Training...

✅ Saved: credit_model.pkl + credit_test.csv
DPD: 0.0177  ✅ Fair
EOD: 0.0113  ✅ Fair

In FairAudit UI:
  Target    → SeriousDlqin2yrs
  Sensitive → age_group  ← NOT age


In [19]:
import pickle, os
import pandas as pd
import numpy as np

# ════════════════════════════════════════════════
# CHANGE ONLY THESE 4 LINES
# ════════════════════════════════════════════════
MODEL_FILE     = "your_model.pkl"       # ← your model file name
TEST_CSV       = "your_test_data.csv"   # ← your test CSV file name
TARGET_COL     = "target"               # ← your target column name
SENSITIVE_COL  = "gender"              # ← your sensitive column name
# ════════════════════════════════════════════════

print("="*65)
print(f"MODEL FILE  : {MODEL_FILE}")
print(f"TEST CSV    : {TEST_CSV}")
print(f"TARGET      : {TARGET_COL}")
print(f"SENSITIVE   : {SENSITIVE_COL}")
print("="*65)

# Check files exist
if not os.path.exists(MODEL_FILE):
    print(f"❌ {MODEL_FILE} not found!")
    print(f"Files available:")
    for f in os.listdir("."):
        if f.endswith(".pkl") or f.endswith(".csv"):
            print(f"  → {f}")
else:
    # Load model
    with open(MODEL_FILE, "rb") as f:
        model = pickle.load(f)
    print(f"✅ Model loaded: {type(model).__name__}")

    # Load test data
    df = pd.read_csv(TEST_CSV)
    print(f"✅ Test data loaded: {len(df):,} rows")
    print(f"   Columns: {df.columns.tolist()}")

    # Prepare
    X_test  = df.drop(columns=[TARGET_COL])
    y_test  = df[TARGET_COL]
    sf_test = df[SENSITIVE_COL]

    # Predictions
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1].round(4)
    else:
        y_proba = np.full(len(y_pred), "N/A")

    # Build results table
    results = pd.DataFrame()
    results[SENSITIVE_COL] = sf_test.values
    results['actual']      = y_test.values
    results['predicted']   = y_pred
    results['probability'] = y_proba
    results['correct']     = (y_test.values == y_pred)
    results['result']      = results.apply(
        lambda r: '✅ Correct'
        if r['correct']
        else ('❌ False Alarm' if r['predicted']==1
        else  '⚠️ Missed'), axis=1
    )

    # ── Sample Predictions ────────────────────────
    print(f"\nSample Predictions (first 20 rows):")
    print("-"*65)
    print(results[[ SENSITIVE_COL, 'actual',
                    'predicted', 'probability',
                    'result']].head(20).to_string(index=False))

    # ── Summary ───────────────────────────────────
    tp  = int(((y_pred==1) & (y_test==1)).sum())
    tn  = int(((y_pred==0) & (y_test==0)).sum())
    fp  = int(((y_pred==1) & (y_test==0)).sum())
    fn  = int(((y_pred==0) & (y_test==1)).sum())
    acc = results['correct'].mean()

    print(f"\nPrediction Summary:")
    print("-"*65)
    print(f"  Total rows         : {len(results):,}")
    print(f"  ✅ Accuracy        : {acc:.1%}")
    print(f"  ✅ True Positives  : {tp:,}")
    print(f"  ✅ True Negatives  : {tn:,}")
    print(f"  ❌ False Positives : {fp:,}")
    print(f"  ⚠️  False Negatives : {fn:,}")

    # ── By Group ──────────────────────────────────
    print(f"\nPredictions by {SENSITIVE_COL} Group:")
    print("-"*65)
    for grp_val in sorted(results[SENSITIVE_COL].unique()):
        grp     = results[results[SENSITIVE_COL] == grp_val]
        grp_acc = grp['correct'].mean()
        grp_sel = (grp['predicted'] == 1).mean()
        grp_tp  = int(((grp['predicted']==1)&(grp['actual']==1)).sum())
        grp_fp  = int(((grp['predicted']==1)&(grp['actual']==0)).sum())
        grp_fn  = int(((grp['predicted']==0)&(grp['actual']==1)).sum())

        print(f"  Group {grp_val} ({len(grp):,} rows):")
        print(f"    Accuracy        : {grp_acc:.1%}")
        print(f"    Selection Rate  : {grp_sel:.1%}")
        print(f"    True  Positives : {grp_tp:,}")
        print(f"    False Positives : {grp_fp:,}")
        print(f"    Missed cases    : {grp_fn:,}")
        print()

    # ── Save predictions ──────────────────────────
    save_name = MODEL_FILE.replace('.pkl','_predictions.csv')
    results.to_csv(save_name, index=False)

    print(f"✅ Predictions saved → {save_name}")
    print(f"📁 Location: {os.path.abspath(save_name)}")

MODEL FILE  : your_model.pkl
TEST CSV    : your_test_data.csv
TARGET      : target
SENSITIVE   : gender
❌ your_model.pkl not found!
Files available:
  → bank_model.pkl
  → bank_test.csv
  → credit_model.pkl
  → credit_test.csv
  → german_credit_model.pkl
  → german_credit_test.csv
  → hr_model.pkl
  → hr_test.csv
  → trained_model.pkl


In [1]:
with open("fairlearn_ui.py", "r", encoding="utf-8") as f:
    code = f.read()

prediction_section = '''
        # ── PREDICTION OUTPUT VIEWER ─────────────────────────
        if st.session_state.data_ready:
            st.markdown("---")
            st.markdown(\'\'\'<div class="shdr">🔍 Model Prediction Output</div>\'\'\',
                       unsafe_allow_html=True)

            if st.button("▶  Generate Predictions on Test Data",
                        key="gen_predictions"):
                try:
                    model   = st.session_state.model_obj
                    df_test = st.session_state.df_test
                    tc      = st.session_state.target_col
                    sc      = st.session_state.sensitive_col

                    X_test  = df_test.drop(columns=[tc])
                    y_test  = df_test[tc]
                    sf_test = df_test[sc]

                    # Get predictions
                    y_pred  = model.predict(X_test)
                    if hasattr(model, "predict_proba"):
                        y_prob = model.predict_proba(X_test)[:,1].round(4)
                    else:
                        y_prob = ["N/A"] * len(y_pred)

                    # Build output table
                    out = pd.DataFrame()
                    out[sc]              = sf_test.values
                    out["Actual"]        = y_test.values
                    out["Predicted"]     = y_pred
                    out["Confidence %"]  = [
                        f"{v*100:.1f}%" if v != "N/A" else "N/A"
                        for v in y_prob
                    ]
                    out["Result"] = out.apply(
                        lambda r:
                        "✅ Correct"     if r["Actual"]==r["Predicted"]
                        else ("❌ False Alarm" if r["Predicted"]==1
                        else  "⚠️ Missed"), axis=1
                    )

                    # Summary cards
                    total   = len(out)
                    correct = (out["Actual"]==out["Predicted"]).sum()
                    tp = int(((y_pred==1)&(y_test==1)).sum())
                    tn = int(((y_pred==0)&(y_test==0)).sum())
                    fp = int(((y_pred==1)&(y_test==0)).sum())
                    fn = int(((y_pred==0)&(y_test==1)).sum())

                    st.markdown("#### Summary")
                    c1,c2,c3,c4,c5 = st.columns(5)
                    c1.markdown(f\'\'\'<div class="mcard">
                        <div class="mval">{total:,}</div>
                        <div class="mlbl">Total Rows</div>
                    </div>\'\'\', unsafe_allow_html=True)
                    c2.markdown(f\'\'\'<div class="mcard">
                        <div class="mval g">{tp:,}</div>
                        <div class="mlbl">✅ True +ve</div>
                    </div>\'\'\', unsafe_allow_html=True)
                    c3.markdown(f\'\'\'<div class="mcard">
                        <div class="mval g">{tn:,}</div>
                        <div class="mlbl">✅ True -ve</div>
                    </div>\'\'\', unsafe_allow_html=True)
                    c4.markdown(f\'\'\'<div class="mcard">
                        <div class="mval o">{fp:,}</div>
                        <div class="mlbl">❌ False Alarm</div>
                    </div>\'\'\', unsafe_allow_html=True)
                    c5.markdown(f\'\'\'<div class="mcard">
                        <div class="mval r">{fn:,}</div>
                        <div class="mlbl">⚠️ Missed</div>
                    </div>\'\'\', unsafe_allow_html=True)

                    st.markdown("---")

                    # Filter options
                    st.markdown("#### Filter & View Predictions")
                    f1,f2,f3 = st.columns(3)
                    with f1:
                        filter_r = st.selectbox(
                            "Filter by Result",
                            ["All","✅ Correct",
                             "❌ False Alarm","⚠️ Missed"],
                            key="pred_filter_r"
                        )
                    with f2:
                        groups = ["All"] + [
                            str(g) for g in
                            sorted(out[sc].unique())
                        ]
                        filter_g = st.selectbox(
                            f"Filter by {sc}",
                            groups,
                            key="pred_filter_g"
                        )
                    with f3:
                        n_rows = st.slider(
                            "Rows to display",
                            10, 500, 50,
                            key="pred_rows"
                        )

                    # Apply filters
                    filtered = out.copy()
                    if filter_r != "All":
                        filtered = filtered[
                            filtered["Result"]==filter_r
                        ]
                    if filter_g != "All":
                        filtered = filtered[
                            filtered[sc].astype(str)==filter_g
                        ]

                    st.markdown(
                        f"Showing **{min(n_rows,len(filtered)):,}**"
                        f" of **{len(filtered):,}** rows"
                    )

                    # Display table
                    st.dataframe(
                        filtered[[sc,"Actual","Predicted",
                                  "Confidence %","Result"
                                ]].head(n_rows)
                                  .reset_index(drop=True),
                        use_container_width=True
                    )

                    # Download button
                    csv = out.to_csv(index=False).encode("utf-8")
                    st.download_button(
                        label="⬇️  Download All Predictions as CSV",
                        data=csv,
                        file_name=f"{st.session_state.model_name}_predictions.csv",
                        mime="text/csv",
                        key="dl_pred"
                    )

                    st.success(
                        f"✅ Predictions generated for "
                        f"{total:,} rows — "
                        f"Accuracy: {correct/total:.1%}"
                    )

                except Exception as e:
                    st.error(f"❌ Prediction failed: {e}")
'''

# Insert after data_ready confirmation button
code = code.replace(
    "        if st.button(\"✅  Confirm — Go to Audit\", key=\"confirm_data\"):",
    "        if st.button(\"✅  Confirm — Go to Audit\", key=\"confirm_data\"):"
)

# Add prediction section before the Step 2 back button
code = code.replace(
    "    n1,_,_ = st.columns([1,3,1])\n    with n1:\n        if st.session_state.model_ready:\n            if st.button(\"← Back to Step 1\", key=\"back1\"):",
    prediction_section + "\n    n1,_,_ = st.columns([1,3,1])\n    with n1:\n        if st.session_state.model_ready:\n            if st.button(\"← Back to Step 1\", key=\"back1\"):"
)

with open("fairlearn_ui.py", "w", encoding="utf-8") as f:
    f.write(code)

print("✅ Prediction Output Viewer added to UI!")
print("Restart Streamlit:")
print("Ctrl+C → streamlit run fairlearn_ui.py")

✅ Prediction Output Viewer added to UI!
Restart Streamlit:
Ctrl+C → streamlit run fairlearn_ui.py
